# 05 — Treino da ablação sem UPOS: cabeçote Biaffine

Variante biaffine da ablação sem UPOS (apenas HEAD + DEPREL), simétrica ao notebook 04.

**Pré-requisitos**: dependências em `../requirements.txt`; corpus Porttinari processado em `PARSEH2IA_DATA/data_dois/complaints_dataset_obj_outxpos` (HuggingFace `datasets`, salvo com `save_to_disk`). Por padrão os caminhos relativos `../../` assumem que este repositório está clonado dentro do diretório de dados (checkpoints e dataset no diretório pai do repositório) — ajuste a célula de configuração se necessário.

*Repositório da dissertação de mestrado — reimplementação do PortParser (ParseH2IA) para o português brasileiro, corpus Porttinari.*

# Ablação Biaffine — DEPREL + HEAD sem UPOS

Treina os 4 modelos com os melhores hiperparâmetros do Optuna (`cv_results_*_biaffine.csv`) mas **sem a cabeça de UPOS**, para avaliar o impacto do POS tagging auxiliar sobre UAS/LAS.

**Modelos:** mBERT · BERTimbau-Base · BERTimbau-Large · ModernJabuticaBERT

In [ ]:
import os, random, numpy as np, torch

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"Seed definida como {seed}")

seed_everything(42)

In [ ]:
import numpy as np
import os, json, gc, shutil
from datetime import datetime
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from typing import Optional

import datasets
from datasets import Dataset, DatasetDict, load_from_disk

from transformers import (
    AutoTokenizer, AutoConfig,
    BertModel, BertPreTrainedModel,
    Trainer, TrainingArguments, EarlyStoppingCallback,
)

## Labels

In [ ]:
DEPREL_LABELS = [
    'det', 'nsubj', 'root', 'obj', 'xcomp', 'punct', 'mark', 'advcl', 'case',
    'obl', 'amod', 'conj', 'cc', 'nmod', 'advmod', 'flat:name', 'ccomp', 'cop',
    'acl', 'nummod', 'acl:relcl', 'ccomp:speech', 'parataxis', 'csubj',
    'aux:pass', 'appos', 'fixed', 'nsubj:pass', 'aux', 'nsubj:outer',
    'obl:agent', 'expl:impers', 'expl', 'discourse', 'orphan', 'dislocated',
    'flat', 'flat:foreign', 'iobj', 'vocative', 'csubj:outer', 'list',
    'reparandum', 'csubj:pass',
]

DEPREL_LABELS_TO_IDX = {l: i for i, l in enumerate(DEPREL_LABELS)}
IDX_TO_DEPREL_LABELS = {i: l for l, i in DEPREL_LABELS_TO_IDX.items()}

print(f"DEPREL labels: {len(DEPREL_LABELS)}")

## Melhores hiperparâmetros (Optuna biaffine)

In [ ]:
# Linha 0 de cada cv_results_*_biaffine.csv = melhor média de LAS no k-fold
BEST_HPS_BIAFFINE = {
    #"google-bert/bert-base-multilingual-cased": {
    #    "learning_rate":    2.963157703469038e-05,
    #    "weight_decay":     0.2375474603771072,
    #    "warmup_ratio":     0.32784860226175966,
    #    "num_train_epochs": 40,
    #},
    #"neuralmind/bert-base-portuguese-cased": {
    #    "learning_rate":    3.122006593110624e-05,
    #    "weight_decay":     0.12885507147896566,
    #    "warmup_ratio":     0.41014432212172464,
    #    "num_train_epochs": 40,
    #},
    #"neuralmind/bert-large-portuguese-cased": {
    #    "learning_rate":    2.999887859421115e-05,
    #    "weight_decay":     0.11648272803187464,
    #    "warmup_ratio":     0.41728983259499797,
    #    "num_train_epochs": 40,
    #},
    "amadeusai/modernJabuticaBERT-Base-1k": {
        "learning_rate":    3.504036658120871e-05,
        "weight_decay":     0.1583486838649066,
        "warmup_ratio":     0.30279577856576295,
        "num_train_epochs": 40,
    },
}

MODELS = list(BEST_HPS_BIAFFINE.keys())
print("Modelos:", MODELS)

## Carregamento dos dados

In [ ]:
import ast

def load_csv_as_hf_dataset(filepath):
    df = pd.read_csv(filepath)
    records = []
    for _, row in df.iterrows():
        records.append({
            'tokens':      ast.literal_eval(row['tokens']),
            'upos':        ast.literal_eval(row['upos']),
            'deprel':      ast.literal_eval(row['deprel']),
            'head_tags':   ast.literal_eval(str(row['head_tags'])),
            'deprel_tags': ast.literal_eval(str(row['deprel_tags'])),
            'upos_tags':   ast.literal_eval(str(row['upos_tags'])),
        })
    return Dataset.from_list(records)

data = DatasetDict({
    'train': load_csv_as_hf_dataset('../../data_dois/train_outxpos.csv'),
    'val':   load_csv_as_hf_dataset('../../data_dois/val_outxpos.csv'),
    'test':  load_csv_as_hf_dataset('../../data_dois/test_outxpos.csv'),
})
data

## Arquitetura Biaffine

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features: int, out_features: int, dropout: float = 0.33):
        super().__init__()
        self.linear     = nn.Linear(in_features, out_features)
        self.activation = nn.ELU()
        self.norm       = nn.LayerNorm(out_features)
        self.dropout    = nn.Dropout(dropout)
        nn.init.orthogonal_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.norm(self.activation(self.linear(x))))


class Biaffine(nn.Module):
    def __init__(self, in_features: int, out_features: int = 1,
                 bias_x: bool = True, bias_y: bool = True):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.bias_x = bias_x
        self.bias_y = bias_y
        self.weight = nn.Parameter(torch.zeros(
            out_features,
            in_features + int(bias_x),
            in_features + int(bias_y),
        ))
        nn.init.normal_(self.weight, std=1.0 / in_features)

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        if self.bias_x:
            x = torch.cat([x, x.new_ones(*x.shape[:-1], 1)], dim=-1)
        if self.bias_y:
            y = torch.cat([y, y.new_ones(*y.shape[:-1], 1)], dim=-1)
        # x: [B, L, H+1]  y: [B, L, H+1]  W: [out, H+1, H+1]
        # result: [B, out, L, L]
        return torch.einsum('bih,ohk,bjk->boij', x, self.weight, y)

In [ ]:
# ── Modelo BERT (ablação: sem UPOS) ──────────────────────────────────────────
class MultiTaskSentencePredictionEncoder(BertPreTrainedModel):
    def __init__(self, config, num_deprel_labels: int,
                 arc_hidden: int = 500, rel_hidden: int = 100, mlp_dropout: float = 0.33):
        super().__init__(config)
        self.num_deprel_labels = num_deprel_labels
        self.arc_hidden = arc_hidden
        self.rel_hidden = rel_hidden
        config.num_deprel_labels = num_deprel_labels
        config.arc_hidden = arc_hidden
        config.rel_hidden = rel_hidden

        self.bert = BertModel(config, add_pooling_layer=False)
        encoder_dropout = (config.classifier_dropout
            if getattr(config, "classifier_dropout", None) is not None
            else config.hidden_dropout_prob)
        self.dropout = nn.Dropout(encoder_dropout)

        self.arc_head_mlp = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.arc_dep_mlp  = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.rel_head_mlp = MLP(config.hidden_size, rel_hidden, mlp_dropout)
        self.rel_dep_mlp  = MLP(config.hidden_size, rel_hidden, mlp_dropout)

        self.arc_biaffine = Biaffine(arc_hidden, out_features=1,            bias_x=True, bias_y=False)
        self.rel_biaffine = Biaffine(rel_hidden, out_features=num_deprel_labels, bias_x=True, bias_y=True)
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, Biaffine):
            nn.init.normal_(module.weight, std=1.0 / module.in_features)
        elif isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias); nn.init.ones_(module.weight)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

    def forward(self, input_ids, attention_mask=None, token_type_ids=None,
                deprel_label=None, head_label=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        seq = self.dropout(outputs.last_hidden_state)
        B, L, _ = seq.shape

        h_arc_dep  = self.arc_dep_mlp(seq)
        h_arc_head = self.arc_head_mlp(seq)
        logits_head = self.arc_biaffine(h_arc_dep, h_arc_head).squeeze(1)  # [B,L,L]

        if self.training and logits_head.abs().max() > 1e4:
            raise RuntimeError("logits_head explodiu — reinicie o kernel.")

        if attention_mask is not None:
            pad_mask = (attention_mask == 0).unsqueeze(1)
            logits_head = logits_head.masked_fill(pad_mask, -1e4)

        h_rel_dep  = self.rel_dep_mlp(seq)
        h_rel_head = self.rel_head_mlp(seq)
        logits_rel = self.rel_biaffine(h_rel_dep, h_rel_head)  # [B,num_deprel,L,L]

        arc_preds = logits_head.argmax(-1).clamp(0, L - 1)
        idx_pred  = arc_preds.unsqueeze(-1).unsqueeze(-1).expand(B, L, 1, self.num_deprel_labels)
        logits_rel_t      = logits_rel.permute(0, 2, 3, 1).contiguous()
        logits_deprel_out = logits_rel_t.gather(2, idx_pred).squeeze(2)

        loss = None
        if head_label is not None and deprel_label is not None:
            loss = self._compute_loss(logits_head, logits_rel, head_label, deprel_label, B, L)

        if loss is not None:
            return (loss, logits_deprel_out, logits_head)
        return (logits_deprel_out, logits_head)

    def _compute_loss(self, logits_head, logits_rel, head_label, deprel_label, B, L):
        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
        oob = (head_label != -100) & (head_label >= L)
        head_c, dep_c = head_label.clone(), deprel_label.clone()
        head_c[oob] = -100; dep_c[oob] = -100

        loss_head = loss_fct(logits_head.reshape(B*L, L), head_c.reshape(-1))

        safe_heads = head_c.clamp(0, L-1)
        idx_gold   = safe_heads.unsqueeze(-1).unsqueeze(-1).expand(B, L, 1, self.num_deprel_labels)
        logits_rel_t      = logits_rel.permute(0, 2, 3, 1).contiguous()
        logits_deprel_gold = logits_rel_t.gather(2, idx_gold).squeeze(2)
        loss_deprel = loss_fct(logits_deprel_gold.reshape(B*L, self.num_deprel_labels), dep_c.reshape(-1))

        for name, val in [("loss_head", loss_head), ("loss_deprel", loss_deprel)]:
            if torch.isnan(val) or torch.isinf(val):
                raise RuntimeError(f"{name}={val.item():.4e} — verifique head_label e logits.")
        return loss_head + loss_deprel

In [ ]:
from transformers import ModernBertModel, ModernBertPreTrainedModel

# ── Modelo ModernBERT (ablação: sem UPOS) ────────────────────────────────────
class MultiTaskSentencePredictionEncoderModern(ModernBertPreTrainedModel):
    def __init__(self, config, num_deprel_labels: int,
                 arc_hidden: int = 500, rel_hidden: int = 100, mlp_dropout: float = 0.33):
        super().__init__(config)
        self.num_deprel_labels = num_deprel_labels
        self.arc_hidden = arc_hidden
        self.rel_hidden = rel_hidden
        config.num_deprel_labels = num_deprel_labels
        config.arc_hidden = arc_hidden
        config.rel_hidden = rel_hidden

        self.model = ModernBertModel(config)
        encoder_dropout = (
            getattr(config, "classifier_dropout", None)
            or getattr(config, "hidden_dropout_prob", None)
            or getattr(config, "embedding_dropout", 0.1)
        )
        self.dropout = nn.Dropout(encoder_dropout)

        self.arc_head_mlp = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.arc_dep_mlp  = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.rel_head_mlp = MLP(config.hidden_size, rel_hidden, mlp_dropout)
        self.rel_dep_mlp  = MLP(config.hidden_size, rel_hidden, mlp_dropout)

        self.arc_biaffine = Biaffine(arc_hidden, out_features=1,                bias_x=True, bias_y=False)
        self.rel_biaffine = Biaffine(rel_hidden, out_features=num_deprel_labels, bias_x=True, bias_y=True)
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, Biaffine):
            nn.init.normal_(module.weight, std=1.0 / module.in_features)
        elif isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias); nn.init.ones_(module.weight)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

    def forward(self, input_ids, attention_mask=None, token_type_ids=None,
                deprel_label=None, head_label=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        seq = self.dropout(outputs.last_hidden_state)
        B, L, _ = seq.shape

        h_arc_dep  = self.arc_dep_mlp(seq)
        h_arc_head = self.arc_head_mlp(seq)
        logits_head = self.arc_biaffine(h_arc_dep, h_arc_head).squeeze(1)

        if self.training and logits_head.abs().max() > 1e4:
            raise RuntimeError("logits_head explodiu — reinicie o kernel.")

        if attention_mask is not None:
            pad_mask = (attention_mask == 0).unsqueeze(1)
            logits_head = logits_head.masked_fill(pad_mask, -1e4)

        h_rel_dep  = self.rel_dep_mlp(seq)
        h_rel_head = self.rel_head_mlp(seq)
        logits_rel = self.rel_biaffine(h_rel_dep, h_rel_head)

        arc_preds = logits_head.argmax(-1).clamp(0, L-1)
        idx_pred  = arc_preds.unsqueeze(-1).unsqueeze(-1).expand(B, L, 1, self.num_deprel_labels)
        logits_rel_t      = logits_rel.permute(0, 2, 3, 1).contiguous()
        logits_deprel_out = logits_rel_t.gather(2, idx_pred).squeeze(2)

        loss = None
        if head_label is not None and deprel_label is not None:
            loss = self._compute_loss(logits_head, logits_rel, head_label, deprel_label, B, L)

        if loss is not None:
            return (loss, logits_deprel_out, logits_head)
        return (logits_deprel_out, logits_head)

    def _compute_loss(self, logits_head, logits_rel, head_label, deprel_label, B, L):
        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
        oob = (head_label != -100) & (head_label >= L)
        head_c, dep_c = head_label.clone(), deprel_label.clone()
        head_c[oob] = -100; dep_c[oob] = -100

        loss_head = loss_fct(logits_head.reshape(B*L, L), head_c.reshape(-1))

        safe_heads = head_c.clamp(0, L-1)
        idx_gold   = safe_heads.unsqueeze(-1).unsqueeze(-1).expand(B, L, 1, self.num_deprel_labels)
        logits_rel_t      = logits_rel.permute(0, 2, 3, 1).contiguous()
        logits_deprel_gold = logits_rel_t.gather(2, idx_gold).squeeze(2)
        loss_deprel = loss_fct(logits_deprel_gold.reshape(B*L, self.num_deprel_labels), dep_c.reshape(-1))

        for name, val in [("loss_head", loss_head), ("loss_deprel", loss_deprel)]:
            if torch.isnan(val) or torch.isinf(val):
                raise RuntimeError(f"{name}={val.item():.4e}")
        return loss_head + loss_deprel

In [ ]:
def build_model(name_model: str, num_deprel_labels: int,
                arc_hidden: int = 500, rel_hidden: int = 100, mlp_dropout: float = 0.33):
    config = AutoConfig.from_pretrained(name_model)
    if config.model_type == "modernbert":
        cls = MultiTaskSentencePredictionEncoderModern
    else:
        cls = MultiTaskSentencePredictionEncoder
    print(f"[build_model] model_type='{config.model_type}' → {cls.__name__}")
    return cls.from_pretrained(
        name_model, config=config,
        num_deprel_labels=num_deprel_labels,
        arc_hidden=arc_hidden, rel_hidden=rel_hidden, mlp_dropout=mlp_dropout,
        _fast_init=False,
    )

## POSDataset — sem UPOS

In [ ]:
class POSDataset:
    def __init__(self, tokenizer_ckpt):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_ckpt)

    def align_labels_with_tokens(self, labels, word_ids):
        new_labels, current_word = [], None
        for word_id in word_ids:
            if word_id != current_word:
                current_word = word_id
                try:
                    label = -100 if word_id is None else labels[word_id]
                except Exception:
                    label = -100
                new_labels.append(label)
            elif word_id is None:
                new_labels.append(-100)
            else:
                new_labels.append(labels[word_id])
        return new_labels

    def preprocess_function(self, examples):
        tokenized_inputs = self.tokenizer(
            examples["tokens"], truncation=True, padding="max_length",
            is_split_into_words=True, max_length=512,
        )
        new_deprel, new_head = [], []
        for i, (dep, head) in enumerate(zip(examples["deprel_tags"], examples["head_tags"])):
            word_ids = tokenized_inputs.word_ids(i)
            new_deprel.append(self.align_labels_with_tokens(dep,  word_ids))
            new_head.append(  self.align_labels_with_tokens(head, word_ids))
        tokenized_inputs["deprel_label"] = new_deprel
        tokenized_inputs["head_label"]   = new_head
        return tokenized_inputs

    def create_data(self, train, test):
        tkn_train = train.map(self.preprocess_function, batched=True, remove_columns=train.column_names)
        tkn_test  = test.map( self.preprocess_function, batched=True, remove_columns=test.column_names)
        return tkn_train, tkn_test

## Data Collator

In [ ]:
def data_collator(batch):
    input_ids       = [item["input_ids"]     for item in batch]
    attention_masks = [item["attention_mask"] for item in batch]
    deprel_label    = [item["deprel_label"]   for item in batch]
    head_label      = [item["head_label"]     for item in batch]
    max_len = max(len(ids) for ids in input_ids)
    PAD = 0
    return {
        "input_ids":      torch.tensor([ids + [PAD]  * (max_len - len(ids)) for ids in input_ids]),
        "attention_mask": torch.tensor([m   + [0]    * (max_len - len(m))   for m   in attention_masks]),
        "deprel_label":   torch.tensor([l   + [-100] * (max_len - len(l))   for l   in deprel_label]),
        "head_label":     torch.tensor([l   + [-100] * (max_len - len(l))   for l   in head_label]),
    }

## Compute Metrics

In [ ]:
import numpy as np
import json
import os

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)


def compute_metrics(eval_pred, PRETRAINED_MODEL=None):
    save_path = f"epoch_predictions/{PRETRAINED_MODEL.split('/')[-1] if PRETRAINED_MODEL else 'unknown_model'}_predictions_ablacao_biaffine.json"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    model_name = PRETRAINED_MODEL.split("/")[-1] if PRETRAINED_MODEL else "unknown_model"
    run_id = model_name

    logits, labels = eval_pred
    deprel_logits, head_logits = logits
    deprel_labels, head_labels = labels

    # ---------------- PROBABILIDADES ----------------
    deprel_probs = softmax(deprel_logits, axis=-1)
    head_probs   = softmax(head_logits,   axis=-1)

    # ---------------- PREDIÇÕES ----------------
    deprel_preds = np.argmax(deprel_logits, axis=-1)
    head_preds   = np.argmax(head_logits,   axis=-1)

    # ---------------- MASK DEPENDENCY ----------------
    valid_mask = (head_labels != -100) & (deprel_labels != -100)

    head_preds_masked    = head_preds[valid_mask]
    head_labels_masked   = head_labels[valid_mask]
    head_probs_masked    = head_probs[valid_mask]
    deprel_preds_masked  = deprel_preds[valid_mask]
    deprel_labels_masked = deprel_labels[valid_mask]
    deprel_probs_masked  = deprel_probs[valid_mask]

    # ---------------- MÉTRICAS PRINCIPAIS ----------------
    uas = (head_preds_masked == head_labels_masked).mean()
    las = (
        (head_preds_masked == head_labels_masked) &
        (deprel_preds_masked == deprel_labels_masked)
    ).mean()

    # ---------------- CARREGAR JSON ----------------
    if os.path.exists(save_path):
        with open(save_path, "r", encoding="utf-8") as f:
            all_data = json.load(f)
    else:
        all_data = {}

    if "META" not in all_data:
        all_data["META"] = {
            "model_name": model_name,
            "pretrained_model": PRETRAINED_MODEL,
            "run_id": run_id,
        }

    if "RANK_PREDICTIONS" not in all_data:
        all_data["RANK_PREDICTIONS"] = {"deprel": [], "head": []}

    def get_correct_rank(prob_vector, correct_label):
        sorted_indices = np.argsort(prob_vector)[::-1]
        return int(np.where(sorted_indices == correct_label)[0][0] + 1)

    # ---------------- DEPREL ----------------
    for i in range(len(deprel_preds_masked)):
        probs         = deprel_probs_masked[i]
        correct_label = int(deprel_labels_masked[i])
        pred_label    = int(deprel_preds_masked[i])
        all_data["RANK_PREDICTIONS"]["deprel"].append({
            "run_id":        run_id,
            "model":         model_name,
            "index":         i,
            "correct_label": correct_label,
            "pred_label":    pred_label,
            "correct_prob":  float(probs[correct_label]),
            "pred_prob":     float(probs[pred_label]),
            "correct_rank":  get_correct_rank(probs, correct_label),
        })

    # ---------------- HEAD ----------------
    for i in range(len(head_preds_masked)):
        probs         = head_probs_masked[i]
        correct_label = int(head_labels_masked[i])
        pred_label    = int(head_preds_masked[i])
        all_data["RANK_PREDICTIONS"]["head"].append({
            "run_id":        run_id,
            "model":         model_name,
            "index":         i,
            "correct_label": correct_label,
            "pred_label":    pred_label,
            "correct_prob":  float(probs[correct_label]),
            "pred_prob":     float(probs[pred_label]),
            "correct_rank":  get_correct_rank(probs, correct_label),
        })

    # ---------------- SALVAR ----------------
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(all_data, f, indent=2, ensure_ascii=False)

    return {"uas": float(uas), "las": float(las)}


## CUDA diagnóstico + WandB

In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

if torch.cuda.is_available():
    t = torch.tensor([1.0], device="cuda")
    assert (t * 2).item() == 2.0
    del t; torch.cuda.empty_cache()
    print(f"✓ Contexto CUDA limpo | GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CPU mode")

os.environ["WANDB_API_KEY"] = "7499f50c2fffad3ce3e34b7b7b02152988a1742c"
import wandb

## Função de treino com HPs fixos (train/val original)

In [ ]:
def train_ablacao_biaffine(name_model, best_hps, results_filename="ablacao_biaffine_results.jsonl"):
    learning_rate    = best_hps["learning_rate"]
    weight_decay     = best_hps["weight_decay"]
    warmup_ratio     = best_hps["warmup_ratio"]
    num_train_epochs = best_hps["num_train_epochs"]

    nerdataset = POSDataset(name_model)
    train_data, valid_data = nerdataset.create_data(data['train'], data['val'])

    print(f"\n{'='*55}")
    print(f"  {name_model}")
    print(f"{'='*55}")

    wandb.init(
        entity="gdlima-universidade-federal-de-pelotas",
        project="hf-optuna",
        name=f"ablacao_biaffine_{name_model.split('/')[-1]}",
        config={
            "learning_rate": learning_rate, "architecture": name_model,
            "epochs": num_train_epochs, "weight_decay": weight_decay,
            "warmup_ratio": warmup_ratio, "ablation": "sem_upos_biaffine",
        },
    )

    model = build_model(name_model, num_deprel_labels=len(DEPREL_LABELS))

    for _n, _p in model.named_parameters():
        if _p.requires_grad and (torch.isnan(_p).any() or torch.isinf(_p).any()):
            raise RuntimeError(f"[CPU] Peso '{_n}' é NaN/Inf antes de mover para CUDA.")

    _device = "cuda" if torch.cuda.is_available() else "cpu"
    model   = model.to(_device)

    output_dir = f"./ablacao_biaffine_{name_model.replace('/','_')}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        fp16=False, bf16=False,
        eval_strategy="epoch",
        learning_rate=learning_rate,
        num_train_epochs=num_train_epochs,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        max_grad_norm=1.0,
        lr_scheduler_type="linear",
        logging_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        #save_only_model=True,
        load_best_model_at_end=True,
        metric_for_best_model="las",
        greater_is_better=True,
        label_smoothing_factor=0.0,
        gradient_checkpointing=False,
        remove_unused_columns=False,
        label_names=["deprel_label", "head_label"],
        report_to="wandb",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        dataloader_num_workers=4,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=valid_data,
        compute_metrics=lambda p: compute_metrics(p, PRETRAINED_MODEL=name_model),
        data_collator=data_collator,
        #callbacks=[EarlyStoppingCallback(5)],
    )

    trainer.train()

    best_path = f"./best_models_ablacao_biaffine/{name_model.replace('/','_')}"
    trainer.save_model(best_path)
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    eval_result = trainer.evaluate()
    uas = eval_result.get("eval_uas")
    las = eval_result.get("eval_las")

    result_dict = {
        "name": name_model,
        "architecture": "biaffine_ablacao_sem_upos",
        "hyperparameters": best_hps,
        "uas": uas, "las": las,
    }

    with open(results_filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(result_dict, ensure_ascii=False) + "\n")

    print(f"  UAS: {uas:.4f} | LAS: {las:.4f}")
    wandb.finish()
    return result_dict


## Treino — todos os modelos

In [ ]:
all_results = {}

for model_name in MODELS:
    print(f"\n{'#'*60}")
    print(f"# ABLAÇÃO BIAFFINE (sem UPOS) — {model_name}")
    print(f"{'#'*60}")
    all_results[model_name] = train_ablacao_biaffine(model_name, BEST_HPS_BIAFFINE[model_name])


## Inferência no conjunto de teste

In [ ]:
dataset_test  = load_from_disk('../../data_dois/complaints_dataset_obj_outxpos')
test_dataset   = dataset_test['test']
test_sentences = test_dataset['tokens']
test_deprel    = test_dataset['deprel']
test_head      = test_dataset['head_tags']

print(f"Test set: {len(test_sentences)} sentenças")

In [ ]:
def get_predictions_on_dataframe(sentences, model, tokenizer, device="cuda"):
    predictions_deprel, predictions_head = [], []

    model.eval()
    model.to(device)

    for tokens in tqdm(sentences):
        inputs = tokenizer(
            tokens, is_split_into_words=True,
            return_tensors="pt", padding=True, truncation=True,
        ).to(device)

        with torch.no_grad():
            model_outputs = model(**inputs)

        logits_deprel = model_outputs[0]
        logits_head   = model_outputs[1]
        word_ids = inputs.word_ids(batch_index=0)

        sent_deprel, sent_head = [], []
        for token_idx in range(len(tokens)):
            sub_idxs = [i for i, w in enumerate(word_ids) if w == token_idx]
            if sub_idxs:
                first = sub_idxs[0]
                prob_d = torch.softmax(logits_deprel[0, first], dim=-1)
                prob_h = torch.softmax(logits_head[0,   first], dim=-1)
                sent_deprel.append(IDX_TO_DEPREL_LABELS[torch.argmax(prob_d).item()])
                sent_head.append(torch.argmax(prob_h).item())

        predictions_deprel.append(sent_deprel)
        predictions_head.append(sent_head)

    return pd.DataFrame({
        "tokens":             sentences,
        "deprel_predictions": predictions_deprel,
        "head_predictions":   predictions_head,
    })

In [ ]:
def compute_dependency_metrics(test_sentences, test_deprel, test_head, predict_df):
    total = uas_correct = las_correct = skipped = 0
    for i in range(len(test_sentences)):
        gold_d = test_deprel[i]
        gold_h = test_head[i]
        pred_d = predict_df['deprel_predictions'].iloc[i]
        pred_h = predict_df['head_predictions'].iloc[i]
        for j in range(len(gold_h)):
            if j >= len(pred_h) or pred_h[j] is None or pred_d[j] is None:
                skipped += 1; continue
            total += 1
            if pred_h[j] == gold_h[j]:
                uas_correct += 1
                if pred_d[j] == gold_d[j]:
                    las_correct += 1
    return {
        'uas': uas_correct / total if total > 0 else 0,
        'las': las_correct / total if total > 0 else 0,
        'total_tokens': total, 'skipped': skipped,
    }

In [ ]:
_device = "cuda" if torch.cuda.is_available() else "cpu"
final_metrics = {}

for model_name in MODELS:
    print(f"\n{'='*50}")
    print(f"Inferência: {model_name}")

    best_model_path = f"./best_models_ablacao_biaffine/{model_name.replace('/','_')}"
    val_las = all_results[model_name]["las"]
    print(f"Modelo salvo em: {best_model_path} | LAS val = {val_las:.4f}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    config    = AutoConfig.from_pretrained(best_model_path)

    if config.model_type == "modernbert":
        model = MultiTaskSentencePredictionEncoderModern.from_pretrained(
            best_model_path, config=config, num_deprel_labels=len(DEPREL_LABELS),
        ).to(_device)
    else:
        model = MultiTaskSentencePredictionEncoder.from_pretrained(
            best_model_path, config=config, num_deprel_labels=len(DEPREL_LABELS),
        ).to(_device)

    predict_df = get_predictions_on_dataframe(test_sentences, model, tokenizer, device=_device)
    metrics    = compute_dependency_metrics(test_sentences, test_deprel, test_head, predict_df)
    final_metrics[model_name] = metrics

    print(f"  UAS: {metrics['uas']:.4f} | LAS: {metrics['las']:.4f}")
    print(f"  Total tokens: {metrics['total_tokens']} | Ignorados: {metrics['skipped']}")

    out_csv = f"./predict_test_ablacao_biaffine_{model_name.replace('/','_')}.csv"
    predict_df.to_csv(out_csv, index=False)
    print(f"  Salvo em: {out_csv}")


## Resumo final

In [ ]:
print("\n" + "="*65)
print("ABLAÇÃO BIAFFINE — DEPREL + HEAD (sem UPOS) — Conjunto de Teste")
print("="*65)
print(f"{'Modelo':<42} {'UAS':>7} {'LAS':>7}")
print("-"*65)
for model_name, metrics in final_metrics.items():
    short = model_name.split('/')[-1]
    print(f"{short:<42} {metrics['uas']:>7.4f} {metrics['las']:>7.4f}")

results_summary = [
    {"model": k, "uas": v["uas"], "las": v["las"],
     "total_tokens": v["total_tokens"], "architecture": "biaffine_ablacao_sem_upos"}
    for k, v in final_metrics.items()
]
pd.DataFrame(results_summary).to_csv("ablacao_biaffine_test_metrics.csv", index=False)
print("\nMétricas salvas em ablacao_biaffine_test_metrics.csv")